In [ ]:
# Clone your repo (you can include -b raph if you want only that branch)
!git clone -b raph --single-branch https://github.com/Rapsim/IPEO_DeepL_group15.git
%cd IPEO_DeepL_group15

# Install Python dependencies from your requirements file
!pip install -r requirements.txt


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os

# This must match the folder you created in Drive
data_root = "/content/drive/MyDrive/IPEO_DeepL_data"
s2_dir = os.path.join(data_root, "S2")
labels_dir = os.path.join(data_root, "labels")

print("S2 dir:", s2_dir)
print("Labels dir:", labels_dir)
print("Number of S2 files:", len([f for f in os.listdir(s2_dir) if f.endswith(".tif")]))
print("Number of label files:", len([f for f in os.listdir(labels_dir) if f.endswith(".tif")]))


In [ ]:
import tifffile
import numpy as np
import matplotlib.pyplot as plt
import os

In [ ]:
# Get one sample file name (first S2 patch)
s2_files = sorted([f for f in os.listdir(s2_dir) if f.endswith(".tif")])
assert len(s2_files) > 0, "No S2 files found in Drive folder!"

sample_name = s2_files[0]
print("Sample file:", sample_name)

s2_path = os.path.join(s2_dir, sample_name)
label_path = os.path.join(labels_dir, sample_name)

# --- Load S2 image ---
img = tifffile.imread(s2_path)  # could be (C,H,W) or (H,W,C)

# Normalize to [0, 1]-ish (Sentinel-2 is typically 0..10000)
img = img.astype(np.float32) / 10000.0

# Ensure shape is (C, H, W)
if img.ndim == 2:
    img = img[np.newaxis, ...]
elif img.ndim == 3:
    if img.shape[0] not in (1, 3, 4, 12, 64) and img.shape[-1] in (1, 3, 4, 12, 64):
        # assume (H, W, C) -> transpose to (C, H, W)
        img = np.transpose(img, (2, 0, 1))
else:
    raise ValueError(f"Unexpected image shape: {img.shape}")

print("Image shape (C,H,W):", img.shape)

# Pick 3 bands for RGB visualization.
# This is just a guess; you can adjust indices if needed.
# Here I use bands 3,2,1 -> indices 2,1,0
c, h, w = img.shape
rgb = np.stack([img[2], img[1], img[0]], axis=-1)  # (H, W, 3)
rgb = np.clip(rgb, 0, 1)

plt.figure(figsize=(5, 5))
plt.imshow(rgb)
plt.axis("off")
plt.title(f"S2 RGB composite: {sample_name}")
plt.show()

# --- Load and show label mask ---
label = tifffile.imread(label_path)
if label.ndim == 3 and 1 in label.shape:
    label = label.squeeze()
print("Label shape:", label.shape)

plt.figure(figsize=(5, 5))
plt.imshow(label)
plt.colorbar()
plt.axis("off")
plt.title(f"Label mask: {sample_name}")
plt.show()
